# RPS Classification에 QAT 적용

## 설명 노트

ResNet50으로 RPS 모델을 다시 학습해보는 노트북입니다.

- 인식 대상: 가위(scissors), 바위(rock), 보(paper)
- 사용 모델: `ResNet50`
- 입력 크기: `64 x 64 x 3`
- 사용 기법: 데이터 보강 + QAT + TFLite 변환
- 내가 찍은 카메라 데이터가 있으면 `RPS_Camera_Dataset/0,1,2`에 넣어서 기존 데이터와 같이 학습
- 최종 산출물: `RPS_PreTrained_ResNet50_CameraFineTuned_QAT.tflite`

목표는 실제 카메라 환경에서 가위바위보 인식률을 조금 더 올려보는 것입니다.


### DenseNet121 버전과 ResNet50 버전의 차이

| 구분 | DenseNet121 버전 | ResNet50 버전 |
|---|---|---|
| Backbone | `keras.applications.DenseNet121` | `keras.applications.ResNet50` |
| 전처리 함수 | `densenet.preprocess_input` | `resnet.preprocess_input` |
| 장점 | 비교적 효율적이고 특징 재사용이 좋음 | 널리 알려진 표준 CNN 구조, 설명이 쉬움 |
| 주의점 | 구조가 초보자에게 다소 낯설 수 있음 | 라즈베리파이에서 속도와 파일 크기가 부담될 수 있음 |

수업에서는 두 모델의 정확도뿐 아니라 `.tflite` 파일 크기, 추론 시간, FPS를 함께 비교하면 좋습니다.


## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택'

In [ ]:
# 필요한 패키지 설치
# Colab, GCP GPU 서버, VSCode Jupyter 모두에서 실행할 수 있게 작성했습니다.
#
# tensorflow-model-optimization: QAT 적용용
# opencv-python-headless: 서버 환경에서 이미지 읽기용. GUI 창이 필요 없으므로 headless 버전을 권장합니다.
# scikit-learn: train/test split용

import sys
import subprocess

required_packages = [
    "tensorflow-model-optimization",
    "opencv-python-headless",
    "scikit-learn",
]

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    *required_packages
], check=True)

print("Package installation/check complete.")


## 모듈 로딩

In [ ]:
# 기본 라이브러리 불러오기
# numpy: 배열 계산
# tensorflow/keras: 딥러닝 모델 생성과 학습
# matplotlib: 결과 그래프와 이미지 확인
# glob: 폴더 안 파일 목록 읽기
# cv2(OpenCV): 이미지 읽기, 색상 변환, 크기 변경
# train_test_split: 학습용/테스트용 데이터 분리
# tfmot: TensorFlow 모델 경량화와 QAT 적용
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import glob
import cv2
from sklearn.model_selection import train_test_split
import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

# 현재 실행 환경의 주요 라이브러리 버전을 확인합니다.
# 버전 차이 때문에 QAT API가 다르게 동작할 수 있어 처음에 확인하는 것이 좋습니다.
print("NumPy Version :{}".format(np.__version__))
print("TensorFlow Version :{}".format(tf.__version__))
print("Matplotlib Version :{}".format(plt.matplotlib.__version__))


### 실행 환경 확인

이 학생용 저장소는 `RPS_Dataset` 폴더를 저장소 안에 포함합니다.
따라서 Google Drive를 연결하지 않아도 Colab, GCP GPU 서버, VSCode에서 같은 방식으로 실행할 수 있습니다.


In [ ]:
# Colab인지, GCP/VSCode 같은 일반 Python 환경인지 확인합니다.
# 데이터셋은 다음 셀에서 저장소 내부의 RPS_Dataset 폴더를 자동으로 찾습니다.
try:
    import google.colab  # type: ignore
    colab = True
    print("Colab environment.")
except Exception:
    colab = False
    print("Local/GCP/VSCode environment.")


### RPS 데이터셋 준비

In [ ]:
# RPS 데이터셋 경로 설정
# 기본 데이터셋은 RPS_Dataset을 사용한다.
# 내가 직접 찍은 사진은 RPS_Camera_Dataset/0,1,2에 넣으면 같이 읽히게 했다.
#
# 폴더 번호:
# 0 -> scissors
# 1 -> rock
# 2 -> paper

import subprocess
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/philipdekim-OnD01/RP2-RPS-QAT-Lab.git"

COLAB_REPO_DIR = Path("/content/RP2-RPS-QAT-Lab")
SERVER_REPO_DIR = Path.home() / "RP2-RPS-QAT-Lab_data"

CLASS_NAMES = {
    0: "scissors",
    1: "rock",
    2: "paper",
}

# 혹시 폴더명을 다르게 만들었을 때도 찾을 수 있게 후보를 몇 개 둔다.
OPTIONAL_CAMERA_DATASET_NAMES = [
    "RPS_Camera_Dataset",
    "Real_RPS_Dataset",
    "My_RPS_Dataset",
]


def get_git_root():
    """지금 노트북이 들어 있는 git 저장소 루트를 찾는다."""
    try:
        result = subprocess.run(
            ["git", "rev-parse", "--show-toplevel"],
            check=True,
            capture_output=True,
            text=True,
        )
        return Path(result.stdout.strip())
    except Exception:
        return None


def clone_repo_if_needed(target_dir):
    """데이터셋을 못 찾았을 때만 원본 저장소를 clone한다."""
    if not target_dir.exists():
        print("Clone GitHub repository for RPS_Dataset:", GITHUB_REPO_URL)
        subprocess.run([
            "git", "clone", "--depth", "1",
            GITHUB_REPO_URL,
            str(target_dir)
        ], check=True)
    else:
        print("Repository already exists:", target_dir)
    return target_dir


def has_class_folders(dataset_dir):
    """0, 1, 2 폴더가 모두 있으면 RPS 데이터셋으로 본다."""
    dataset_dir = Path(dataset_dir)
    return all((dataset_dir / str(ind)).is_dir() for ind in CLASS_NAMES)


def image_count(dataset_dir, class_id):
    """각 클래스에 이미지가 몇 장 있는지 확인한다."""
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    class_dir = Path(dataset_dir) / str(class_id)
    if not class_dir.is_dir():
        return 0
    return sum(1 for p in class_dir.iterdir() if p.is_file() and p.suffix.lower() in image_exts)


candidate_paths = []

git_root = get_git_root()
if git_root is not None:
    candidate_paths.append(git_root / "RPS_Dataset")

cwd = Path.cwd()
candidate_paths.extend([
    cwd / "RPS_Dataset",
    cwd / "../RPS_Dataset",
    Path("RPS_Dataset"),
    Path("../RPS_Dataset"),
])

primary_dataset_path = None
for candidate in candidate_paths:
    candidate = candidate.resolve()
    if candidate.exists() and candidate.is_dir():
        primary_dataset_path = candidate
        break

if primary_dataset_path is None:
    clone_dir = COLAB_REPO_DIR if colab else SERVER_REPO_DIR
    repo_dir = clone_repo_if_needed(clone_dir)
    dataset_dir = repo_dir / "RPS_Dataset"
    if not dataset_dir.exists():
        raise FileNotFoundError(
            f"RPS_Dataset 폴더를 찾을 수 없습니다: {dataset_dir}. "
            "GITHUB_REPO_URL이 올바른지 확인하세요."
        )
    primary_dataset_path = dataset_dir

# 기존 코드에서 files_path를 쓰는 부분이 있을 수 있어서 남겨둔다.
files_path = str(primary_dataset_path)

# 추가로 찍은 카메라 데이터셋을 찾아본다.
search_roots = []
for root in [git_root, primary_dataset_path.parent, cwd, cwd.parent, Path.home()]:
    if root is not None:
        resolved = Path(root).resolve()
        if resolved not in search_roots:
            search_roots.append(resolved)

additional_dataset_paths = []
for root in search_roots:
    for name in OPTIONAL_CAMERA_DATASET_NAMES:
        candidate = (root / name).resolve()
        if candidate == primary_dataset_path.resolve():
            continue
        if candidate.exists() and has_class_folders(candidate):
            if candidate not in additional_dataset_paths:
                additional_dataset_paths.append(candidate)

# 실제 학습에는 기본 데이터셋 + 내가 찍은 데이터셋을 같이 사용한다.
dataset_paths = [primary_dataset_path] + additional_dataset_paths

print("Working directory:", Path.cwd())
print("Git root:", git_root)
print("Primary RPS dataset path:", primary_dataset_path)

if additional_dataset_paths:
    print("Additional camera dataset paths:")
    for dataset_dir in additional_dataset_paths:
        print(" -", dataset_dir)
else:
    print("Additional camera dataset paths: none found yet")
    print("카메라 데이터가 준비되면 RPS_Camera_Dataset/0,1,2 폴더에 넣으면 된다.")

print("Dataset folders used for training:")
for dataset_dir in dataset_paths:
    print("-", dataset_dir)
    for ind, name in CLASS_NAMES.items():
        print(f"  class {ind} ({name}): {image_count(dataset_dir, ind)} images")


### 폴더 별로 파일 읽어 데이터화 진행

In [ ]:
%%time
# 이미지 파일을 읽어서 학습용 numpy 배열로 만든다.
# 기본 RPS_Dataset과 내가 추가한 카메라 데이터셋을 같이 읽는다.

first = True
IMG_SIZE = 64
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

if "dataset_paths" not in globals():
    dataset_paths = [Path(files_path)]


def list_image_files(class_dir):
    """이미지 파일만 골라서 정렬한다."""
    class_dir = Path(class_dir)
    if not class_dir.is_dir():
        return []
    return sorted([
        p for p in class_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ])


def read_and_resize_images(files):
    """OpenCV로 이미지를 읽고 64x64 RGB로 맞춘다."""
    images = []
    skipped = []
    for image_path in files:
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            skipped.append(image_path)
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
        images.append(image)
    if skipped:
        print("Skipped unreadable files:")
        for image_path in skipped[:10]:
            print(" -", image_path)
        if len(skipped) > 10:
            print(" ...", len(skipped) - 10, "more")
    return np.array(images, dtype=np.uint8)


for dataset_dir in dataset_paths:
    dataset_dir = Path(dataset_dir)
    print("Reading dataset:", dataset_dir)

    for ind in range(0, 3, 1):
        class_dir = dataset_dir / str(ind)
        files = list_image_files(class_dir)
        print(f"  class {ind} ({CLASS_NAMES[ind]}): {len(files)} files from {class_dir}")

        if len(files) == 0:
            print("  -> no files, skipped")
            continue

        tmpx = read_and_resize_images(files)

        if len(tmpx) < 2:
            print("  -> train/test로 나누려면 읽을 수 있는 이미지가 최소 2장 필요해서 건너뜀")
            continue

        tmpy = np.array([ind] * len(tmpx), dtype=np.int64)

        # 각 데이터셋과 클래스마다 80:20으로 나눈다.
        # 실제 최종 테스트 이미지는 가능하면 따로 남겨두는 게 더 좋다.
        xtrain, xtest, ytrain, ytest = train_test_split(
            tmpx, tmpy, test_size=0.2, random_state=123
        )

        if first == True:
            train_data = xtrain.copy()
            train_labels = ytrain.copy()
            test_data = xtest.copy()
            test_labels = ytest.copy()
            first = False
        else:
            train_data = np.concatenate((train_data, xtrain))
            train_labels = np.concatenate((train_labels, ytrain))
            test_data = np.concatenate((test_data, xtest))
            test_labels = np.concatenate((test_labels, ytest))

if first:
    raise RuntimeError("읽을 수 있는 이미지가 없습니다. RPS_Dataset이나 카메라 데이터셋 폴더를 확인하세요.")


### 데이터 확인

In [ ]:
# 데이터 모양(shape)을 확인합니다.
# 예상 형태:
# train_data   -> (학습 이미지 수, 64, 64, 3)
# train_labels -> (학습 라벨 수,)
# test_data    -> (테스트 이미지 수, 64, 64, 3)
# test_labels  -> (테스트 라벨 수,)

# ResNet의 preprocess_input()에서 입력 전처리를 처리합니다.
# 그래서 여기서는 train_data / 255.0 같은 정규화를 하지 않습니다.
# train_data = train_data / 255.0
# test_data = test_data / 255.0

print(train_data.shape)
print(train_labels.shape)
print(test_data.shape)
print(test_labels.shape)


### ImageDataGenerator 객체 생성

In [ ]:
# ImageDataGenerator는 학습 이미지를 매번 조금씩 변형해 주는 도구입니다.
# 목적: 실제 카메라 환경처럼 손 위치, 밝기, 각도, 크기가 달라져도 잘 맞히게 만들기
img_gen_train = tf.keras.preprocessing.image.ImageDataGenerator(
                horizontal_flip=True,         # 좌우 반전. 손 모양 방향이 바뀌는 상황 대응
                rotation_range=0.2,           # 이미지를 조금 회전. 손이 기울어진 상황 대응
                width_shift_range=0.1,        # 좌우 이동. 손이 중앙에서 벗어난 상황 대응
                height_shift_range=0.1,       # 상하 이동. 손 높이가 달라지는 상황 대응
                brightness_range=(0.1, 0.9),  # 밝기 변화. 조명이 어둡거나 밝은 상황 대응
                zoom_range=0.2                # 확대/축소. 카메라와 손 거리 변화 대응
                )

# flow()는 원본 train_data를 읽어, 학습 중에 변형된 이미지를 계속 공급합니다.
# seed를 고정하면 실험 재현성이 좋아집니다.
train_gen = img_gen_train.flow(train_data, train_labels, seed = 1)


### 모델 생성


In [ ]:
# 모델 생성 단계
# 이 버전은 DenseNet이 아니라 ResNet50을 사용합니다.
# ResNet50은 ImageNet으로 미리 학습된 이미지 인식 모델입니다.
# 우리는 맨 뒤 분류기를 가위/바위/보 3개 클래스로 바꿔서 사용합니다.

# 입력 이미지는 64 x 64 RGB 이미지입니다.
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# ResNet이 기대하는 방식으로 입력 이미지를 전처리합니다.
# ResNet50은 ImageNet 학습 때 사용한 전처리 기준을 맞춰 주는 것이 중요합니다.
x = keras.applications.resnet.preprocess_input(inputs)

# ImageNet 사전학습 가중치를 가진 ResNet50을 불러옵니다.
# include_top=False는 ImageNet용 1000개 클래스 분류 머리를 제거한다는 뜻입니다.
# input_tensor=x로 우리가 만든 입력과 전처리 흐름을 연결합니다.
base_model = keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    input_tensor=x
)

# BatchNormalization layer는 학습 중 통계가 흔들리면 전이학습 성능이 불안정할 수 있어 고정합니다.
# 나머지 layer는 trainable=True로 두어 RPS 데이터에 맞게 미세조정합니다.
for layer in base_model.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

# ResNet50이 뽑은 특징맵을 하나의 벡터로 요약합니다.
x = base_model.output
x = keras.layers.GlobalAveragePooling2D()(x)

# Dropout은 과적합을 줄이기 위해 일부 뉴런을 학습 중 임의로 꺼 줍니다.
x = keras.layers.Dropout(0.2)(x)

# 최종 출력은 3개입니다.
# 0: scissors, 1: rock, 2: paper
outputs = keras.layers.Dense(3, activation='softmax')(x)

# 입력부터 출력까지 연결한 최종 모델입니다.
model = keras.Model(inputs, outputs)

# 모델 구조와 파라미터 수를 확인합니다.
# ResNet50은 DenseNet121보다 파라미터 수와 TFLite 파일 크기가 커질 수 있습니다.
model.summary()


In [ ]:
# QAT(Quantization Aware Training) 적용 준비
# QAT는 나중에 TFLite 양자화 모델로 바꿀 때 정확도 손실을 줄이기 위한 학습 방법입니다.
# 학습 중에 int8 양자화로 생길 오차를 미리 흉내 내면서 모델을 적응시킵니다.

# 선택적 양자화 함수
# 모든 layer를 양자화하지 않고, 주로 계산량이 큰 Dense와 Conv2D layer에만 표시합니다.
def apply_quantization_to_layer(layer):
    # Dense와 Conv2D 레이어만 양자화 대상이라고 표시합니다.
    if isinstance(layer, (keras.layers.Dense, keras.layers.Conv2D)):
        return tfmot.quantization.keras.quantize_annotate_layer(layer)

    # 나머지 layer는 그대로 둡니다.
    return layer

# 기존 model을 복제하면서 위 함수가 지정한 layer에 양자화 annotation을 붙입니다.
annotated_model = keras.models.clone_model(
    model,
    clone_function=apply_quantization_to_layer,
)

# annotation을 실제 QAT layer로 변환합니다.
# 이 결과가 학습에 사용할 qat_model입니다.
qat_model = tfmot.quantization.keras.quantize_apply(annotated_model)


In [ ]:
# QAT가 적용된 모델 구조를 확인합니다.
# Conv2D나 Dense layer 주변에 QuantizeWrapperV2가 보이면 QAT 적용이 된 것입니다.
qat_model.summary()


### 모델 컴파일

In [ ]:
# 모델 학습 설정
# optimizer: Adam 사용
# learning_rate=1e-5: 사전학습 모델을 미세조정하므로 학습률을 작게 둡니다.
# loss: sparse_categorical_crossentropy
#   라벨이 one-hot이 아니라 0, 1, 2 정수이기 때문에 sparse 버전을 사용합니다.
# metrics: accuracy로 정확도를 확인합니다.
qat_model.compile(optimizer= keras.optimizers.Adam(learning_rate=1e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


### 학습 전 상황

In [ ]:
# 예측 결과를 눈으로 확인하기 위한 함수입니다.
# 테스트 이미지 10장을 골라 실제 라벨과 모델 예측값을 제목에 표시합니다.
def Make_Result_Plot(suptitle, data, label, y_max):
    # 테스트 데이터를 10개 구간으로 나누어 대표 이미지를 하나씩 봅니다.
    size = data.shape[0]//10

    # 2행 5열 그림판을 만듭니다.
    fig_result, ax_result = plt.subplots(2,5,figsize=(18, 7))
    fig_result.suptitle(suptitle)

    for idx in range(10):
        cnt = idx * size + size//2

        # 이미지 출력
        ax_result[idx//5,idx%5].imshow(data[cnt].reshape((IMG_SIZE,IMG_SIZE, 3)))

        # label: 실제 정답, y: 모델 예측값
        ax_result[idx//5,idx%5].set_title(
            "test_data[{}] (label : {} / y : {})".format(cnt, label[cnt], y_max[cnt])
        )


In [ ]:
# 학습 전 모델의 예측 상태를 확인합니다.
# 아직 RPS 데이터에 맞게 충분히 학습되지 않았기 때문에 틀리는 예측이 많을 수 있습니다.
y_out = qat_model.predict(test_data)

# softmax 출력 3개 중 가장 큰 값의 위치를 예측 클래스로 선택합니다.
y_max = np.argmax(y_out, axis=1).reshape((-1, 1))

# 학습 전 예측 결과를 그림으로 확인합니다.
Make_Result_Plot("Before Training", test_data, test_labels, y_max)


### 콜백 설정

In [ ]:
# 학습 중 validation 성능이 가장 좋은 모델만 저장한다.
# 카메라 데이터를 추가한 실험이라 파일명에 CameraFineTuned를 붙였다.
savedModelName = 'RPS_PreTrained_ResNet50_CameraFineTuned_QAT.h5'

callbacks = [
    keras.callbacks.ModelCheckpoint(savedModelName,
                                    save_best_only=True)
]


### 모델 학습

In [ ]:
%%time
# 모델 학습
# train_gen은 ImageDataGenerator가 만든 변형 이미지를 계속 공급합니다.
# epochs=10은 전체 학습 데이터를 10번 반복해서 본다는 뜻입니다.
# validation_data는 학습에 쓰지 않은 test_data로 성능을 확인합니다.
history = qat_model.fit(train_gen, epochs=10,
                    callbacks=callbacks,
                    validation_data=(test_data, test_labels))


> ### Ploting : Cost/Training Count

In [ ]:
# 학습 과정 시각화
# 위 그래프: 정확도 변화
# 아래 그래프: 손실(loss) 변화
# train과 validation이 함께 좋아지면 학습이 잘 되고 있는 것입니다.

plt.figure(figsize=(6, 10))

# 정확도 그래프
plt.subplot(2, 1, 1)
plt.title('Accuray')
plt.plot(history.history['accuracy'], 'b', label='train_accuracy')
plt.plot(history.history['val_accuracy'], 'g', label='val_accuracy')
# plt.ylim([0,1])
plt.grid(True)
plt.ylabel('Accuracy')
plt.legend(loc='best')

# 손실 그래프
plt.subplot(2, 1, 2)
plt.title('Loss')
plt.plot(history.history['loss'], 'b', label='train_loss')
plt.plot(history.history['val_loss'], 'g', label='val_loss')
# plt.ylim([0,5])
plt.grid(True)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='best')
plt.show()


### best 모델 로딩 및 테스트

In [ ]:
# 저장된 best 모델 다시 불러오기
# QAT 모델에는 QuantizeWrapperV2 같은 특수 layer가 들어 있습니다.
# 그래서 일반 load_model이 아니라 quantize_scope 안에서 불러와야 합니다.
from tensorflow_model_optimization.quantization.keras import quantize_scope

try:
    # quantize_scope() 안에서 모델을 로드하면 QAT 관련 사용자 정의 객체를 인식합니다.
    with quantize_scope():
        model_best = keras.models.load_model(savedModelName)
except ValueError as e:
    print(f"Model load failed: {e}")


In [ ]:
# 학습 후 best 모델의 예측 결과를 확인합니다.
y_out = model_best.predict(test_data)
y_max = np.argmax(y_out, axis=1).reshape((-1, 1))
Make_Result_Plot("After Training", test_data, test_labels, y_max)


### best 모델 백업

In [ ]:
# 학습된 모델과 TFLite 파일을 저장할 위치를 정합니다.
# 학생용 저장소에서는 results/ 폴더에 결과를 저장합니다.
# results/는 .gitignore에 들어 있으므로 큰 모델 파일이 실수로 push되지 않습니다.
from pathlib import Path

try:
    repo_root = get_git_root()
except NameError:
    repo_root = None

if repo_root is None:
    repo_root = Path.cwd()

save_dir = str(repo_root / "results") + "/"
Path(save_dir).mkdir(parents=True, exist_ok=True)
print("save_dir:", save_dir)


In [ ]:
# 학습된 best Keras 모델(.h5)을 results/ 폴더로 복사합니다.
# results/는 .gitignore에 포함되어 있어 GitHub에는 올라가지 않습니다.
import shutil
from pathlib import Path

src_model = Path(savedModelName)
dst_model = Path(save_dir) / savedModelName

if src_model.exists():
    shutil.copy2(src_model, dst_model)
    print("Saved best model to:", dst_model)
else:
    print("Model file not found:", src_model)


### 초보자 핵심 정리

여기까지의 핵심은 다음과 같습니다.

1. `RPS_Dataset`에서 가위/바위/보 이미지를 읽어 `64 x 64 x 3` 배열로 만듭니다.
2. ImageNet으로 미리 학습된 `ResNet50`을 가져와 마지막 분류기를 3개 클래스로 바꿉니다.
3. 데이터 보강으로 조명, 위치, 회전, 크기 변화에 강하게 만듭니다.
4. QAT를 적용해 TFLite 양자화 후에도 정확도가 유지되도록 학습합니다.
5. 최종 `.tflite` 파일을 라즈베리파이에 복사해 실시간 손 모양 인식에 사용합니다.

주의: ResNet50은 실습 비교용으로 좋지만 라즈베리파이에서는 DenseNet121 또는 MobileNet 계열보다 느릴 수 있습니다. 모델 정확도뿐 아니라 FPS와 지연 시간도 함께 비교해야 합니다.


### LiteRT 모델로 변환

In [ ]:
# Keras QAT 모델을 TFLite 모델로 변환합니다.
# TFLite는 라즈베리파이 같은 엣지 장치에서 실행하기 좋은 모델 형식입니다.
converter = tf.lite.TFLiteConverter.from_keras_model(model_best)

# Optimize.DEFAULT는 기본 양자화/최적화를 적용하라는 뜻입니다.
# QAT로 학습했기 때문에 양자화 후 정확도 손실을 줄이는 데 도움이 됩니다.
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 실제 변환 실행. 결과는 bytes 형태의 tflite_model입니다.
tflite_model = converter.convert()


### LiteRT 모델을 파일로 저장

In [ ]:
# 라즈베리파이에 가져갈 TFLite 파일 이름
# 카메라 데이터 추가 실험 결과라 CameraFineTuned 이름을 붙였다.
tfliteFileName = 'RPS_PreTrained_ResNet50_CameraFineTuned_QAT.tflite'


In [ ]:
# 변환된 TFLite 모델을 파일로 저장합니다.
# ResNet50은 DenseNet121보다 파일 크기가 커질 수 있습니다.
# 저장 후 실제 파일 크기와 라즈베리파이 추론 속도를 반드시 확인하세요.
open(save_dir + tfliteFileName, 'wb').write(tflite_model)
